<a href="https://colab.research.google.com/github/stiltnerag/INFO648/blob/main/Homework_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 ***Part 1***

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split #split our data for us
from sklearn.compose import ColumnTransformer #help us with onehot encoding
from sklearn.preprocessing import OneHotEncoder #ovy
from sklearn.linear_model import LinearRegression #our model
from sklearn.pipeline import Pipeline #our model becomes more than regession - workflow
from sklearn.metrics import mean_absolute_error, root_mean_squared_error #metrics



In [10]:
deals=pd.read_csv('/content/saas_deals.csv')

In [7]:
deals.head()
deals.describe()
deals.describe(include='all')

,deal_id,company_employees,sales_calls,pipeline_days,industry,plan_tier,region,acv_usd
count,300,300.000000,300.000000,300.000000,300,300,300,300.000000
unique,300,NaN,NaN,NaN,5,3,3,NaN
top,D-0173,NaN,NaN,NaN,Tech,Pro,NorthAmerica,NaN
freq,1,NaN,NaN,NaN,80,144,170,NaN
mean,NaN,1177.373333,15.923333,92.843333,NaN,NaN,NaN,69109.313333
std,NaN,1717.906689,8.889368,48.290566,NaN,NaN,NaN,34415.112768
min,NaN,16.000000,1.000000,5.000000,NaN,NaN,NaN,5000.000000
25%,NaN,62.000000,8.000000,55.500000,NaN,NaN,NaN,43925.500000
50%,NaN,303.500000,17.000000,93.000000,NaN,NaN,NaN,60715.500000
75%,NaN,1675.250000,24.000000,130.500000,NaN,NaN,NaN,90609.750000


#Numeric                               

*   company_employees
*   sales_calls
*   pipeline_days
*   acv_usd

#Categorical


*   industry
*   plan_tier
*   region








#Part 2 - Train/test split

In [19]:
X = deals[['company_employees','sales_calls','pipeline_days', 'industry','plan_tier','region']]
y = deals['acv_usd']
#Split the data into test and train
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=0)

In [20]:
X_test.shape

(75, 6)

We set aside a test before fitting the model to evaluate how well our model performs on new data. We want to make sure that the machine isn't memorizing the random noise and createing false illusions of high accuracy.

#Part 3  – Build the Pipeline

In [21]:
numeric_cols     =['company_employees','sales_calls','pipeline_days']
categorical_cols =['industry','plan_tier','region']
preprocessor = ColumnTransformer(
    transformers=[
        ('cat',
         OneHotEncoder(handle_unknown='ignore'),
         categorical_cols),
        ('num', 'passthrough', numeric_cols),
    ]
)

In [22]:
model=Pipeline([
    ('prep',preprocessor),
    ('lr',LinearRegression())
])

model

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['industry', 'plan_tier',
                                                   'region']),
                                                 ('num', 'passthrough',
                                                  ['company_employees',
                                                   'sales_calls',
                                                   'pipeline_days'])])),
                ('lr', LinearRegression())])

#Part 4 -  Fit and evaluate

In [23]:
model.fit(X_train,y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['industry', 'plan_tier',
                                                   'region']),
                                                 ('num', 'passthrough',
                                                  ['company_employees',
                                                   'sales_calls',
                                                   'pipeline_days'])])),
                ('lr', LinearRegression())])

In [25]:
y_pred=model.predict(X_test)
preview=pd.DataFrame({'Actual':y_test,'Predict':y_pred})
preview


,Actual,Predict
208,50509,45152.670831
188,41939,32543.782317
12,106150,120961.820871
221,52338,61499.723921
239,98709,79724.898795
...,...,...
156,39558,29355.963809
228,40941,47278.719709
273,93516,78679.348647
27,55676,57135.300841


In [33]:
mae=mean_absolute_error(y_test,y_pred)
rmse=root_mean_squared_error(y_test,y_pred)
print(f': MAE=${mae:,.2f} and RMSE=${rmse:,.2f}')

: MAE=$10,626.68 and RMSE=$13,444.11


On average our model is off by about $10,626 per deal.

#Part 5 - Interpret the coefficients

In [38]:
# Pull feature names out of the preprocessor and pair them with coefficients
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(drop='first', sparse_output=False)
feature_names = model.named_steps['prep'].get_feature_names_out()
coefs = model.named_steps['lr'].coef_
intercept = model.named_steps['lr'].intercept_

coef_table = pd.DataFrame({
    'feature': feature_names,
    'coefficient_$K': coefs.round(2),
})
print(f'intercept = ${intercept*1000:,.0f}')
coef_table

intercept = $55,780,147


,feature,coefficient_$K
0,cat__industry_Finance,12478.07
1,cat__industry_Healthcare,10114.03
2,cat__industry_Manufacturing,-4352.89
3,cat__industry_Retail,-11440.48
4,cat__industry_Tech,-6798.74
5,cat__plan_tier_Basic,-32889.61
6,cat__plan_tier_Enterprise,47317.27
7,cat__plan_tier_Pro,-14427.66
8,cat__region_APAC,434.47
9,cat__region_EMEA,4932.37


1. Numeric Coefficient:
    Each additional sales call a rept makes is associated with an increase of  $666.51 in annual contract value.

2. Categorical Coefficient: If a sales rep gets an Enterprise contract it is associated with a massive revenue bump of $47,317.27 in annual contract value.

3. Numeric Coefficient: Each additional day a deal is in the pipeline there is an increase of $47.80 in the final annual contract value.

#Part 6 - Bonus:

In [39]:
np.random.seed(42)

In [41]:
deals['lucky_number'] = np.random.randint(1, 101, size=len(deals))
deals['paperclips_sold'] = np.random.randint(0, 51, size=len(deals))
deals['office_temperature'] = np.random.randint(60, 91, size=len(deals))

In [42]:
print(deals[['lucky_number', 'paperclips_sold', 'office_temperature']].head())

   lucky_number  paperclips_sold  office_temperature
0            52               15                  76
1            93                2                  66
2            15               19                  83
3            72               23                  82
4            61               32                  89


In [43]:
X = deals.drop(columns=["deal_id", "acv_usd"])
y = deals["acv_usd"]

In [44]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [45]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

In [48]:
from sklearn.preprocessing import MinMaxScaler
numeric_transformer = Pipeline(steps=[("scaler", MinMaxScaler())])

In [51]:
from sklearn.linear_model import LassoCV

lasso_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("estimator", LassoCV(cv=5, random_state=0)),
    ]
)

In [52]:
lasso_pipeline.fit(X_train, y_train)

print("LassoCV Pipeline fitted successfully!")

LassoCV Pipeline fitted successfully!


In [53]:
feature_names = lasso_pipeline.named_steps['preprocessor'].get_feature_names_out()

In [54]:
coefficients = lasso_pipeline.named_steps['estimator'].coef_

In [55]:
lasso_coef_table = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
})

In [56]:
kept_features = lasso_coef_table[lasso_coef_table['coefficient'] != 0]
dropped_features = lasso_coef_table[lasso_coef_table['coefficient'] == 0]

print("=== FEATURES KEPT (Non-Zero Coefficients) ===")
print(kept_features.to_string(index=False))

print("\n=== FEATURES DROPPED (Coefficients Equal to 0) ===")
print(dropped_features.to_string(index=False))

=== FEATURES KEPT (Non-Zero Coefficients) ===
                  feature  coefficient
cat__plan_tier_Enterprise 15954.813647
   num__company_employees     3.349766
         num__sales_calls   158.758296
       num__pipeline_days    90.513331

=== FEATURES DROPPED (Coefficients Equal to 0) ===
                    feature  coefficient
      cat__industry_Finance          0.0
   cat__industry_Healthcare          0.0
cat__industry_Manufacturing         -0.0
       cat__industry_Retail         -0.0
         cat__industry_Tech         -0.0
       cat__plan_tier_Basic         -0.0
         cat__plan_tier_Pro         -0.0
           cat__region_APAC          0.0
           cat__region_EMEA          0.0
   cat__region_NorthAmerica         -0.0


Yes, all junk features for lucky number, paperclips sold and office temperature was removed.

In [57]:
y_pred_lasso = lasso_pipeline.predict(X_test)

In [58]:
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)

In [59]:
print("=== TEST MAE COMPARISON ===")
print(f"Original Linear Regression MAE : $10,626.68")
print(f"New LassoCV Regression MAE     : ${mae_lasso:,.2f}")
print("===========================")

=== TEST MAE COMPARISON ===
Original Linear Regression MAE : $10,626.68
New LassoCV Regression MAE     : $24,872.54


The new LassoCV MAE is higher than that of the orginial linear regression.